# Checking the engineered tabular features

This notebook compares `data/processed/tabular_clean` with the output of
`src/build_tabular_features.py`. Each section ends with a `PASS` check.

Run `python src/build_tabular_features.py` first so that `data/processed/tabular_features`
and `data/processed/tabular_features_folds` exist.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

pd.set_option("display.max_columns", 20, "display.width", 200)

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
KEYS = ["nodeID", "Year"]

clean = pd.read_parquet(ROOT / "data" / "processed" / "tabular_clean")
log = pd.read_csv(ROOT / "data" / "processed" / "cleaning_log.csv")
splits = pd.read_csv(ROOT / "data" / "splits" / "temporal_splits.csv")
features = pd.read_parquet(ROOT / "data" / "processed" / "tabular_features")
feat_log = pd.read_csv(ROOT / "data" / "processed" / "feature_engineering_log.csv")

print("clean:", clean.shape, " features:", features.shape, " dropped-column log:", feat_log.shape)
MIN_SCALE = 1e-2  # matches src/build_tabular_features.py

clean: (33664, 669)  features: (33664, 1240)  dropped-column log: (196, 5)


## 1. No company-years lost, added, or relabelled

Feature engineering should only change and add columns. The set of company-years must
be identical to `tabular_clean`.

In [2]:
clean_keys = clean[KEYS].sort_values(KEYS, ignore_index=True)
feat_keys = features[KEYS].sort_values(KEYS, ignore_index=True)
assert clean_keys.equals(feat_keys), "company-years differ between tabular_clean and tabular_features"
assert not features.duplicated(KEYS).any(), "duplicate company-year rows in tabular_features"
print(f"PASS: {len(features):,} company-years match exactly")

PASS: 33,664 company-years match exactly


## 2. Every dropped column has |corr| > 0.95 with its matched column

Recomputed independently from `tabular_clean`, not reusing anything the script computed internally. 

A drop's `matched_column` is the column it was directly compared against. `survivor_column` is
resolved by following `matched_column` forward through any further drops, to the column
that actually remains in `tabular_features`

In [3]:
sample = feat_log.sample(min(30, len(feat_log)), random_state=0) if len(feat_log) > 30 else feat_log
recomputed = [
    abs(clean[row.dropped_column].corr(clean[row.matched_column]))
    for row in sample.itertuples()
]
sample = sample.assign(recomputed_corr=np.round(recomputed, 4))
mismatches = sample[(sample["recomputed_corr"] - sample["correlation"]).abs() > 1e-3]
assert mismatches.empty, f"correlation mismatch:\n{mismatches}"
assert (feat_log["correlation"].abs() > 0.95).all(), "a logged drop is not actually above threshold"

kept_numeric = set(log.loc[log['action'] == 'kept', 'column']) - {'IndustryCodeC'}
dropped_set = set(feat_log['dropped_column'])
survivor_set = set(feat_log['survivor_column'])
final_numeric_cols = set(features.columns) & kept_numeric
assert dropped_set <= kept_numeric, "a dropped column was not a kept numeric column to begin with"
assert dropped_set.isdisjoint(final_numeric_cols), "a dropped column is still present in tabular_features"
assert survivor_set <= final_numeric_cols, "a survivor_column is not actually present in tabular_features"
print(f"PASS: {len(sample)} sampled drops match their recomputed correlation")
print(f"PASS: {len(dropped_set)} columns dropped, {len(final_numeric_cols)} numeric columns survive")

PASS: 30 sampled drops match their recomputed correlation
PASS: 196 columns dropped, 469 numeric columns survive


## 3. YoY delta spot check

One company, a couple of columns, checked by hand against `value[year] - value[year-1]`.
Every surviving numeric column except the two geographic ones gets
`sign(x) * log1p(|x|)` applied before the YoY delta, so the manual check below applies
the same transform to both example columns before diffing.

In [4]:
groups = log.set_index("column")["group"]
company = "L000002"
check_cols = ["A001101000", "F010101A"]
GEOGRAPHIC = ("RegisterLongitude", "RegisterLatitude")

rows = clean[clean["nodeID"] == company].sort_values("Year")[["Year"] + check_cols].set_index("Year")
transformed = rows.copy()
for c in check_cols:
    if c not in GEOGRAPHIC:
        transformed[c] = np.sign(transformed[c]) * np.log1p(transformed[c].abs())
manual_yoy = transformed.diff()

yoy_cols = [f"{c}_yoy" for c in check_cols]
actual = features[features["nodeID"] == company].sort_values("Year").set_index("Year")[yoy_cols]
actual.columns = check_cols

compare = manual_yoy.notna() & actual.notna()
assert np.allclose(manual_yoy[compare], actual[compare], equal_nan=True), "YoY delta does not match a manual diff"
assert (manual_yoy.isna() == actual.isna()).all().all(), "YoY NaN pattern does not match a manual diff"
print(f"PASS: YoY deltas for {company} match value[year] - value[year-1] (transformed, both columns)")
pd.concat({"raw": rows, "manual_yoy": manual_yoy, "stored_yoy": actual}, axis=1)

PASS: YoY deltas for L000002 match value[year] - value[year-1] (transformed, both columns)


raw           manual_yoy           stored_yoy          
        A001101000  F010101A A001101000  F010101A A001101000  F010101A
Year                                                                  
2014  6.271525e+10  1.344714        NaN       NaN        NaN       NaN
2015  5.318038e+10  1.302247  -0.164915 -0.018278  -0.164915 -0.018278
2016  8.703212e+10  1.243616   0.492588 -0.025797   0.492588 -0.025797
2017  1.741210e+11  1.200857   0.693473 -0.019242   0.693473 -0.019242
2018  1.884174e+11  1.154342   0.078909 -0.021361   0.078909 -0.021361
2019  1.661946e+11  1.130738  -0.125501 -0.011017  -0.125501 -0.011017
2020  1.952307e+11  1.174494   0.161023  0.020328   0.161023  0.020328
2021  1.493524e+11  1.220232  -0.267873  0.020816  -0.267873  0.020816
2022  1.372076e+11  1.313188  -0.084814  0.041015  -0.084814  0.041015

## 4. No leakage: per-fold imputation and scaling used train rows only

For one fold, independently refit `SimpleImputer` + `RobustScaler` on exactly that
fold's training rows (pulled from `tabular_features`) and confirm transforming all rows reproduces the stored file exactly. If
the script had accidentally fit on train+test, this would not match.

In [5]:
s = 3
fold_rows = splits.loc[splits["split"] == s, ["nodeID", "Year", "role"]]
df = fold_rows.merge(features, on=KEYS, how="left", validate="one_to_one")
feature_cols = [c for c in df.columns if c not in ("nodeID", "Year", "role")]
train = (df["role"] == "train").to_numpy()

imputer = SimpleImputer(strategy="median").fit(df.loc[train, feature_cols])
scaler = RobustScaler().fit(imputer.transform(df.loc[train, feature_cols]))
scaler.scale_[scaler.scale_ < MIN_SCALE] = 1.0  # same near-zero-IQR guard as build_tabular_features.py
recomputed = scaler.transform(imputer.transform(df[feature_cols])).astype(np.float32)

stored = pd.read_parquet(ROOT / "data" / "processed" / "tabular_features_folds" / f"split_{s}")
stored = fold_rows[KEYS].merge(stored, on=KEYS, how="left")[feature_cols].to_numpy()

assert np.allclose(recomputed, stored, atol=1e-4), "stored values do not match a train-only refit — possible leakage"
print(f"PASS: split {s}'s stored values exactly match an independent train-only refit ({train.sum():,} train rows)")

PASS: split 3's stored values exactly match an independent train-only refit (28,532 train rows)


## 5. Industry one-hot columns sum to exactly 1 per row

In [6]:
industry_cols = [c for c in features.columns if c.startswith("industry_")]
sums = features[industry_cols].sum(axis=1)
assert (sums == 1).all(), "a row does not have exactly one industry flag set"
print(f"PASS: all {len(features):,} rows have exactly one of {len(industry_cols)} industry flags set")

PASS: all 33,664 rows have exactly one of 81 industry flags set


## 6. Missing-indicator flags match the real NaN pattern

Checked against `tabular_clean` directly

In [7]:
was_missing_cols = [c for c in features.columns if c.endswith("_was_missing")]
original_cols = [c[: -len("_was_missing")] for c in was_missing_cols]
assert set(original_cols) <= set(clean.columns), "a _was_missing flag does not map back to a tabular_clean column"

clean_sorted = clean.sort_values(KEYS, ignore_index=True)
feat_sorted = features.sort_values(KEYS, ignore_index=True)
actual_na = clean_sorted[original_cols].isna()
flags = feat_sorted[was_missing_cols].astype(bool)
flags.columns = original_cols
assert (actual_na == flags).all().all(), "a _was_missing flag disagrees with the real NaN pattern"

first_year = feat_sorted["is_first_year_on_record"].astype(bool)
true_first_year = feat_sorted.sort_values(KEYS).groupby("nodeID").cumcount() == 0
assert (first_year.to_numpy() == true_first_year.to_numpy()).all(), "is_first_year_on_record is wrong for some row"
print(f"PASS: {len(was_missing_cols)} missing-indicator flags match tabular_clean's real NaN pattern")
print("PASS: is_first_year_on_record correctly marks each company's first row")

PASS: 220 missing-indicator flags match tabular_clean's real NaN pattern
PASS: is_first_year_on_record correctly marks each company's first row
